import

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

**1-Load the data using pd.read_csv() and print the first 5 rows**

to take a overview at the dataset



In [2]:
df = pd.read_csv('/content/employee2.csv')
print(df.head(5))



    age   salary  years_experience  weekly_hours  performance_score   bonus  \
0  56.0   7000.0          1.548044          57.0               3.76  4104.0   
1  46.0   8052.0          7.464551          34.0               1.80  1169.0   
2  32.0  missing         16.391317          53.0               3.14  4864.0   
3  60.0   8387.0         11.555457          54.0               1.39  1783.0   
4  25.0  missing          6.267442          44.0               2.80  1423.0   

   projects_handled department education_level employment_type region  
0               NaN         HR             PhD        Contract  North  
1               7.0      Sales          Master       Full-time  North  
2               1.0  Marketing     High School       Part-time   West  
3              17.0         HR        Bachelor        Contract   West  
4               9.0         IT     High School       Full-time  North  


**Check the shape: how many rows and columns?**

to take a overview at the dataset and knowing the number of columns before cleaning and after .

In [3]:
df_shape = df.shape
print(df_shape) #there are 520 rows and 11 columns

(520, 11)


**Inspect data types: using .info() — are any columns the wrong type? Fixing at least 2**


The wrong type of data leads to flaws in analysis and prediction.
also normalize missing values in real word to NaN (missing in the computer)

In [4]:
print(df.info())

target = 'salary'
#changin any element in age column that is not a number
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df[target] = pd.to_numeric(df[target], errors="coerce")



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                493 non-null    object 
 1   salary             496 non-null    object 
 2   years_experience   494 non-null    float64
 3   weekly_hours       493 non-null    float64
 4   performance_score  493 non-null    float64
 5   bonus              494 non-null    float64
 6   projects_handled   494 non-null    float64
 7   department         493 non-null    object 
 8   education_level    495 non-null    object 
 9   employment_type    494 non-null    object 
 10  region             495 non-null    object 
dtypes: float64(5), object(6)
memory usage: 44.8+ KB
None


**Find missing values: using .isnull().sum() — which columns have the most? Decide what to do (fill in or drop) and explain why**

i choose filling because the missings are few .

i find and fill the  missing values because missing values Can dramatically skew predictions

In [5]:
missing = df.isnull().sum() # The sum of the missing values
print(missing , '\n the age column is most column that has missing values')

#fill the missing values with mean of the values
df["age"] = df["age"].fillna(df["age"].mean())
df["years_experience"] = df["years_experience"].fillna(df["years_experience"].mean())#fill the missing values with mean of the values
df["weekly_hours"] = df["weekly_hours"].fillna(df["weekly_hours"].mean())
df["performance_score"] = df["performance_score"].fillna(df["performance_score"].mean())
df['projects_handled'] = df['projects_handled'].fillna(df['projects_handled'].mean())
#fill the missing values with mode of the values
df["department"] = df["department"].fillna(df["department"].mode()[0])
df["education_level"] = df["education_level"].fillna(df["education_level"].mode()[0])
df["employment_type"] = df["employment_type"].fillna(df["employment_type"].mode()[0])
df["region"] = df["region"].fillna(df["region"].mode()[0])
#fill the missing values with median of the values
df['bonus'] = df['bonus'].fillna(df['bonus'].median())
df[target] = df[target].fillna(df[target].median())




age                  37
salary               34
years_experience     26
weekly_hours         27
performance_score    27
bonus                26
projects_handled     26
department           27
education_level      25
employment_type      26
region               25
dtype: int64 
 the age column is most column that has missing values


**Unifying different names**


Because the computer perceives them as two different things, when in fact they are one and the same. This negatively impacts the analytical prediction.

In [6]:
canonical_map = {"bachelor": "Bachelor",'sales':'Sales'}#Unifying different names, such as: "bachelor": "Bachelor"
#Application of unification of names
df["education_level"] = df["education_level"].map(canonical_map).fillna(df["education_level"])
df["department"] = df["department"].map(canonical_map).fillna(df["department"])



**Handle duplicates: checking with .duplicated().sum() and removing if any**


Because it distorts the data and adds new information that wasn't there.

For example, saying that two people have the same traits is rare and difficult to find.

In [7]:
mask_full = df.duplicated().sum()#check if ther is duplicates
print(mask_full)
df =  df.drop_duplicates()# remove duplicates

18


**Spot outliers: using a the IQR method on the target column; caping extreme values at the
99th percentile**

to prevent these extreme values
from distorting the analysis.

Why IQR

• Based on quartiles, not mean/std

• Less affected by extreme values themselves



In [8]:
# IQR application
q1, q3 = df[target].quantile(0.25), df[target].quantile(0.75)# Determine q1,q3
iqr = q3 -q1
lower  = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
mask_iqr = df[(df[target] < lower) | (df[target] > upper)]
print(mask_iqr)

upper_99 = df[target].quantile(0.99)# determine extreme values at the 99th percentile
df[target] = df[target].clip(upper=upper_99)#cap extreme values at the 99th percentile

           age   salary  years_experience  weekly_hours  performance_score  \
15   20.000000  64085.0          1.689430     27.000000               3.16   
113  24.000000  70015.0          4.048619     31.000000               2.47   
229  28.000000  45450.0          6.839857     45.000000               4.68   
252  18.000000  67390.0         19.522131     39.206897               3.03   
296  33.000000  98080.0         23.250943     57.000000               1.01   
450  46.000000  91640.0          8.890313     39.206897               4.48   
464  40.991718  90570.0         15.019558     55.000000               2.52   
481  27.000000  81120.0          1.091331     22.000000               4.43   

      bonus  projects_handled department education_level employment_type  \
15   3569.0              18.0  Marketing             PhD        Contract   
113  3947.0               9.0    Finance          Master       Full-time   
229   919.0               6.0      Sales          Master       Full-t

**To make it easier to use every time and to avoid rewriting the code.**

In [9]:
df = pd.read_csv('/content/employee2.csv')

def clean_data(df):
  df_copy = df.copy()
  print(df_copy.head(5))
  print(df_copy.info())
  df_shape = df.shape
  print(df_shape) #there are 520 rows and 11 columns


  target = 'salary'
  #changin any element in age column that is not a number
  df_copy["age"] = pd.to_numeric(df_copy["age"], errors="coerce")
  df_copy[target] = pd.to_numeric(df_copy[target], errors="coerce")


  df_copy = df_copy.replace(["unknown",'missing'], np.nan)#replace words represent missing (in real word) to missing values in computer
  missing = df_copy.isnull().sum() # The sum of the missing values
  print(missing , '\n the age column is most column that has missing values')

  #fill the missing values with mean of the values
  df_copy["age"] = df_copy["age"].fillna(df_copy["age"].mean())
  df_copy["years_experience"] = df_copy["years_experience"].fillna(df_copy["years_experience"].mean())#fill the missing values with mean of the values
  df_copy["weekly_hours"] = df_copy["weekly_hours"].fillna(df_copy["weekly_hours"].mean())
  df_copy["performance_score"] = df_copy["performance_score"].fillna(df_copy["performance_score"].mean())
  df_copy['projects_handled'] = df_copy['projects_handled'].fillna(df_copy['projects_handled'].mean())
  #fill the missing values with mode of the values
  df_copy["department"] = df_copy["department"].fillna(df_copy["department"].mode()[0])
  df_copy["education_level"] = df_copy["education_level"].fillna(df_copy["education_level"].mode()[0])
  df_copy["employment_type"] = df_copy["employment_type"].fillna(df_copy["employment_type"].mode()[0])
  df_copy["region"] = df_copy["region"].fillna(df_copy["region"].mode()[0])
  #fill the missing values with median of the values
  df_copy['bonus'] = df_copy['bonus'].fillna(df_copy['bonus'].median())
  df_copy[target] = df_copy[target].fillna(df_copy[target].median())



  canonical_map = {"bachelor": "Bachelor",'sales':'Sales'}#Unifying different names, such as: "bachelor": "Bachelor"
  #Application of unification of names
  df_copy["education_level"] = df_copy["education_level"].map(canonical_map).fillna(df_copy["education_level"])
  df_copy["department"] = df_copy["department"].map(canonical_map).fillna(df_copy["department"])


  mask_full = df_copy.duplicated().sum()#check if ther is duplicates
  print(mask_full)
  df_copy =  df_copy.drop_duplicates()# remove duplicates



  # IQR application
  q1, q3 = df_copy[target].quantile(0.25), df_copy[target].quantile(0.75)# Determine q1,q3
  iqr = q3 -q1
  lower  = q1 - 1.5 * iqr
  upper = q3 + 1.5 * iqr
  mask_iqr = df_copy[(df_copy[target] < lower) | (df_copy[target] > upper)]
  print(mask_iqr)

  upper_99 = df_copy[target].quantile(0.99)# determine extreme values at the 99th percentile
  df_copy[target] = df_copy[target].clip(upper=upper_99)#cap extreme values at the 99th percentile

  #checking

  # check if the target column is under 0
  for i in df_copy[target]:
    if i  < 0 :
      print('there is value under 0 in target column')

  # check if there are null values in the age and target column
  null_values = df_copy[['age', target]].isnull() # seeing wher is the null values in df_copy

  if True not in null_values :
      print('there is no null value in the target column and the age column')


  # check if the number of column is the same befor cleaning and afte
  df_copy_shape = df_copy.shape

  for i in df_shape :
    pass
  for j in df_copy_shape:
    pass
    if j == i :
      print(' the number of columns are the same')




  return df_copy

df =clean_data(df)




    age   salary  years_experience  weekly_hours  performance_score   bonus  \
0  56.0   7000.0          1.548044          57.0               3.76  4104.0   
1  46.0   8052.0          7.464551          34.0               1.80  1169.0   
2  32.0  missing         16.391317          53.0               3.14  4864.0   
3  60.0   8387.0         11.555457          54.0               1.39  1783.0   
4  25.0  missing          6.267442          44.0               2.80  1423.0   

   projects_handled department education_level employment_type region  
0               NaN         HR             PhD        Contract  North  
1               7.0      Sales          Master       Full-time  North  
2               1.0  Marketing     High School       Part-time   West  
3              17.0         HR        Bachelor        Contract   West  
4               9.0         IT     High School       Full-time  North  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 1